# Normalize read-based data to to gene length, and create combined anndata object
We apply gene-length normalization because the number of reads representing a transcript is proportional to the lenght of the transcript, unless UMIs are used.

Run in terminal

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis
# python -m ipykernel install --user --name scrna_cartography_py_analysis --display-name "py_analysis"

Load Packages

In [1]:
# https://docs.scvi-tools.org/en/stable/tutorials/notebooks/scrna/tabula_muris.html
import os
import sys
import glob
from pyhere import here

import anndata
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import torch
import seaborn as sns
from plotnine import*

# My modules / functions
sys.path.append(str(here('scripts/misc')))  
import my_anndata as ma

Setup Parameters

In [2]:
# Create folder if it does not excist
ma.create_directories(dir_path = str(here('data/anndata/preprocess')))

# Paths to data files
metadata_paths = glob.glob(str(here('data/anndata/metadata') / '*_metadata.csv'))
anndata_dirs = glob.glob(str(here('data/anndata/preprocess') / '*.h5ad'))

# Plot dir
plot_dir = str(here('data/misc_plots'))

/work/islet_cartography_scrna/data/anndata/preprocess Directory already exists!


In [3]:
# Plotting
sc.set_figure_params(figsize=(5, 5), frameon=False)
sns.set_theme(style="white", palette=None)
torch.set_float32_matmul_precision("high")
sc.settings.figdir = plot_dir
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

Copy raw anndata files to 'data/anndata/integration', because we want to load data using back-end mode, so we are making changes directly

In [ ]:
# !cp /work/islet_cartography_scrna/data/anndata/raw/*.h5ad /work/islet_cartography_scrna/data/anndata/preprocess/ 

Load data

In [ ]:
# Gene lenghts
gene_len = pd.read_csv(str(here('genome_files/gene_lengths.csv')), 
                           delimiter = ',',
                          index_col = 0)

Normalize count-based methods to gene length

In [4]:
# Create an anndata dictionary
adata_dict = {}

# Add study name to dictionary and add anndata file path
for file_path in anndata_dirs:
    key = Path(file_path).stem.replace(".h5ad", "")
    adata_dict[key] = file_path

In [ ]:
for sample_name, h5ad_path in adata_dict.items():
    
    # Load the AnnData object as backend
    adata = sc.read_h5ad(h5ad_path, backed = 'r')
    
    if 'count_quantification' in adata.obs.columns:

        quant_type = adata.obs['count_quantification'].iloc[0]
        
        if quant_type == 'read_based':

            # Load full adata object into memory
            adata = sc.read_h5ad(h5ad_path)

            print(f"Normalizing '{sample_name}' for gene length...")
            
            # Store index in sets
            initial_genes = set(adata.var.index)
            genes_with_lengths = set(gene_len.index)
            
            # Compare difference 
            missing_genes = initial_genes - genes_with_lengths
        
            if missing_genes: # This checks if the set is not empty
                raise ValueError(f"Gene length data is missing for {len(missing_genes)} genes in AnnData object. "
                                 f"Please ensure all genes in your count matrix have a corresponding length in the gene_len file.")

            # Make index the same order 
            gene_len = gene_len.reindex(adata.var.index)
            
            assert (adata.var.index == gene_len.index).sum() == adata.shape[1]
    
            # Normalize 
            normalized_X = adata.X / gene_len['mean_gene_len'].values * np.median(gene_len['mean_gene_len'].values)
            
            # round to integer
            normalized_X = np.rint(normalized_X)
    
            # Convert to sparse format
            adata.X = sparse.csr_matrix(normalized_X)

            # Rounding to the nearest integer. This change is also written directly to the file.
            adata.write(h5ad_path)
        
    elif quant_type == 'umi_based':
        print(f"Skipping normalization for '{sample_name}' as it's UMI-based.")
    
    else:
        print(f"Unknown quantification type for '{sample_name}': {quant_type}")  

Concatinate the dataset without loading into memory

In [7]:
anndata.experimental.concat_on_disk(
    in_files=adata_dict,         
    out_file=str(here("data/anndata/A_combined.h5ad")), # output file
    axis=0 # concatenate along observations (This stacks the datasets vertically, adding more observations (cells))
)